# Edit Search

Run bounded search over direction layers, application spans, and edit strengths on the selection split.

**Runtime:** T4 GPU

In [1]:
!nvidia-smi

Sat Apr 18 10:42:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             50W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import os
import subprocess
import sys
from getpass import getpass
from pathlib import Path

try:
    from google.colab import userdata as colab_userdata
except ImportError:
    colab_userdata = None


def read_secret(name: str) -> str:
    if colab_userdata is not None:
        try:
            return colab_userdata.get(name).strip()
        except Exception:
            pass
    return os.environ.get(name, "").strip()


REPO_URL = "https://github.com/aaliyan1230/false-refusal-suppression.git"

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    try:
        HF_TOKEN = read_secret("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass("Enter your HF token (or press Enter to skip): ")

REPO_DIR = Path("/content/false-refusal-suppression")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "HEAD"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train,dev]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "transformers>=4.52", "accelerate"], check=True)

os.chdir(REPO_DIR)
src_path = REPO_DIR / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import transformers
print(f"transformers version: {transformers.__version__}")
print(REPO_DIR)
print("HF token loaded:", bool(HF_TOKEN))

transformers version: 5.5.4
/content/false-refusal-suppression
HF token loaded: True


In [ ]:
from pathlib import Path

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

# --- XSTest benchmark splits (250 safe borderline + 200 unsafe contrast) ---
SPLIT_DIR = Path("data/processed/splits/xstest")

ACTIVATION_ARTIFACT = Path("artifacts/activations/llama31_8b_xstest.json")
DIRECTION_ARTIFACT = Path("artifacts/directions/llama31_8b_xstest_unsafe_vs_easy.json")
SEARCH_ARTIFACT = Path("artifacts/edits/llama31_8b_xstest_search.json")

required_paths = [SPLIT_DIR / "discovery.jsonl", SPLIT_DIR / "selection.jsonl"]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing XSTest splits (should be in repo): {missing}")

print(MODEL_ID)
print(f"Split: {SPLIT_DIR}")
print("Direction artifact exists:", DIRECTION_ARTIFACT.exists())
print("Search artifact target:", SEARCH_ARTIFACT)

CalledProcessError: Command '['/usr/bin/python3', 'scripts/convert_xstest.py']' returned non-zero exit status 2.

In [9]:
import subprocess, sys

# Step 1: Measure activations (if missing)
if not ACTIVATION_ARTIFACT.exists():
    print("=== Measuring activations (this loads the model once) ===", flush=True)
    cmd = [
        sys.executable, "scripts/measure_activations.py",
        "--model-id",   MODEL_ID,
        "--split-path",  str(SPLIT_DIR / "discovery.jsonl"),
        "--output",      str(ACTIVATION_ARTIFACT),
        "--group", "unsafe_true_refusal",
        "--group", "benign_easy",
        "--capture-default-modules",
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError("measure_activations.py failed")
    print("✓ Activations measured")
else:
    print("Activations already exist, skipping")

# Step 2: Compute directions (if missing)
if not DIRECTION_ARTIFACT.exists():
    print("\n=== Computing contrast directions ===", flush=True)
    cmd = [
        sys.executable, "scripts/compute_directions.py",
        "--activations",    str(ACTIVATION_ARTIFACT),
        "--source-group-a", "unsafe_true_refusal",
        "--source-group-b", "benign_easy",
        "--output",         str(DIRECTION_ARTIFACT),
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError("compute_directions.py failed")
    print("✓ Directions computed")
else:
    print("Directions already exist, skipping")

print(f"\nDirection artifact: {DIRECTION_ARTIFACT} ({DIRECTION_ARTIFACT.stat().st_size} bytes)")

=== Measuring activations (this loads the model once) ===
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).

Fetching 4 files: 100%|██████████| 4/4 [00:36<00:00,  9.02s/it]

Loading weights: 100%|██████████| 291/291 [00:04<00:00, 60.06it/s]
artifacts/activations/llama31_8b_gemini_pilot.json
✓ Activations measured

=== Computing contrast directions ===
artifacts/directions/llama31_8b_gemini_pilot_unsafe_vs_easy.json
✓ Directions computed

Direction artifact: artifacts/directions/llama31_8b_gemini_pilot_unsafe_vs_easy.json (4001455 bytes)


In [10]:
import subprocess, sys, threading

cmd = [
    sys.executable,
    "scripts/search_edits.py",
    "--model-id",       MODEL_ID,
    "--direction-artifact", str(DIRECTION_ARTIFACT),
    "--selection-split", str(SPLIT_DIR / "selection.jsonl"),
    "--output",         str(SEARCH_ARTIFACT),
    "--top-k-layers",   "3",
    "--strength",  "0.5",
    "--strength",  "1.0",
    "--span-width", "1",
    "--module-type", "attn_out",
    "--module-type", "mlp_down",
    "--write-partial-results",
]
print("Running:", " ".join(cmd), flush=True)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

# Stream output line-by-line so progress is visible
for line in proc.stdout:
    print(line, end="", flush=True)

rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"search_edits.py failed with exit code {rc}")
print("\n✓ Edit search completed")

Running: /usr/bin/python3 scripts/search_edits.py --model-id meta-llama/Llama-3.1-8B-Instruct --direction-artifact artifacts/directions/llama31_8b_gemini_pilot_unsafe_vs_easy.json --selection-split data/processed/splits/pilot_gemini/selection.jsonl --output artifacts/edits/llama31_8b_gemini_pilot_search.json --top-k-layers 3 --strength 0.5 --strength 1.0 --span-width 1 --module-type attn_out --module-type mlp_down --write-partial-results
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).

Loading weights: 100%|██████████| 291/291 [00:04<00:00, 67.82it/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loaded 32 selection prompts
Base evaluation complete; evaluating 18 edit candidates
[1/18] source=layer_31 layers=[31] modules=('attn_out',) strength=0.5 norm_preserving=False
Completed [1/18] in 110.0s: false_refusal=0.00

KeyboardInterrupt: 

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

with open(SEARCH_ARTIFACT, "r", encoding="utf-8") as handle:
    candidates = json.load(handle)

df = pd.DataFrame(candidates)
display(df.head(10))

if not df.empty:
    ax = df.plot.scatter(
        x="false_refusal_rate",
        y="true_refusal_rate",
        s=df["capability_retention"].clip(lower=0.05) * 300,
        figsize=(6, 6),
        title="Edit search candidates",
    )
    ax.set_xlabel("false refusal rate")
    ax.set_ylabel("true refusal rate")
    plt.show()

In [ ]:
# Push artifacts to GitHub so notebook 30 can pull them
import subprocess
subprocess.run(["git", "add", str(SEARCH_ARTIFACT), str(DIRECTION_ARTIFACT),
                "data/processed/splits/xstest/"], check=True)
subprocess.run(["git", "commit", "-m", "artifact: XSTest edit search results + splits"], check=True)
subprocess.run(["git", "push", "origin", "main"], check=True)
print(f"✓ Pushed artifacts to GitHub")